# Replicate GameInterpreter experiment with Copilot premium models

Models:
- "gpt-5"
- "claude-sonnet-4.5"

> Note: readthedocs will not be able to run copilot. Need to find a way to not run but save outputs.

In [1]:
from IPython.display import Markdown, display, HTML
from draw_tree import draw_tree, generate_tikz, generate_png
import urllib.request
import json
from urllib.parse import quote
import subprocess

<!-- Display image from web -->
<img src="https://raw.githubusercontent.com/zczlsde/GameInterpreter/main/Image/Full_Pipeline.png" alt="Full Pipeline" width="75%">

Function to load game descriptions and prompts from GitHub repository:

In [2]:
def get_descriptions(folder):
    api_url = "https://api.github.com/repos/zczlsde/GameInterpreter/contents/" + folder
    url_base = "https://raw.githubusercontent.com/zczlsde/GameInterpreter/main/" + folder + "/"

    # Use GitHub API to list files in the directory
    with urllib.request.urlopen(api_url) as response:
        files_info = json.loads(response.read().decode('utf-8'))

    # Get list of .txt files
    txt_files = [f['name'] for f in files_info if f['name'].endswith('.txt')]

    print(f"Found {len(txt_files)} text files in {folder}:")
    print(txt_files)

    # Load descriptions
    descriptions = {}
    for filename in txt_files:
        with urllib.request.urlopen(url_base + quote(filename)) as response:
            descriptions[filename] = response.read().decode('utf-8')
    return descriptions

Function to call Copilot CLI with specified model and prompt:

> Note: might be better to use OpenAI Python SDK directly since copilot only has premium models (e.g. GPT-5) and the most recent GameInterpreter paper used GPT5-mini because this performed as well as GPT-5.

In [3]:
def call_copilot(model, prompt):
    cmd = f"copilot --model {model} -p '{prompt}'"
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if proc.stdout:
        return proc.stdout
    else:
        # Return error message if no output
        print("STDERR:", proc.stderr)
        print("Return code:", proc.returncode)

Get a dictionary with all the prompts:

In [4]:
prompts = get_descriptions("Prompts")

Found 5 text files in Prompts:
['Code_Generation_Initialization.txt', 'EFG_Generation.txt', 'Error_Message.txt', 'Imperfect_Information_Retrieval.txt', 'Imperfect_Information_Retrieval_Initialization.txt']


This description of the API in the prompt is wrong, so change it here:

In [5]:
prompts["Code_Generation_Initialization.txt"] = prompts["Code_Generation_Initialization.txt"].replace(
    "efg = g.write(format='native')",
    "g.to_efg('games/one-shot-trust-game-after-kreps-1990.efg')"
    )

In [6]:
display(Markdown(prompts["Code_Generation_Initialization.txt"]))

Given a game description in natural language, you will be asked to generate python code for the Gambit API (pygambit) to construct a corresponding extensive-form game in Gambit. 
Here are two examples of how to use pygambit library:

Example 1:
Game description:
There are two players, a Buyer and a Seller. The Buyer moves first and has two actions, Trust or Not trust. If the Buyer chooses Not trust, then the game ends, and both players receive payoffs of 0. If the Buyer chooses Trust, then the Seller has a choice with two actions, Honor or Abuse. If the Seller chooses Honor, both players receive payoffs of 1; if the Seller chooses Abuse, the Buyer receives a payoff of -1 and the Seller receives a payoff of 2.
Code:
```python
import pygambit as gbt
g = gbt.Game.new_tree(players=["Buyer", "Seller"],
                    title="One-shot trust game, after Kreps (1990)")

g.append_move(g.root, "Buyer", ["Trust", "Not trust"])
g.append_move(g.root.children[0], "Seller", ["Honor", "Abuse"])
g.set_outcome(g.root.children[0].children[0], g.add_outcome([1, 1], label="Trustworthy"))
g.set_outcome(g.root.children[0].children[1], g.add_outcome([-1, 2], label="Untrustworthy"))
g.set_outcome(g.root.children[1], g.add_outcome([0, 0], label="Opt-out"))
# Save the EFG
g.to_efg('games/one-shot-trust-game-after-kreps-1990.efg')
```

Example 2:
Game description:
There are two players, Alice and Bob. There is a deck of cards, with equal numbers of King and Queen cards. The game begins with each player putting $1 in the pot. One card is dealt at random to Alice; Alice observes her card but Bob does not. After Alice observes her card, she can choose either to Raise or to Fold. If she chooses to Fold, Bob wins the pot and the game ends. If she chooses to Raise, she adds another $1 to the pot. Bob then chooses either to Meet or Pass. If he chooses to Pass, Alice wins the pot and the game ends. If he chooses to Meet, he adds another $1 to the pot. There is then a showdown, in which Alice reveals her card. If she has a King, then she wins the pot; if she has a Queen, then Bob wins the pot.
Code:
```python
import pygambit as gbt
g = gbt.Game.new_tree(players=["Alice", "Bob"],
                    title="One card poker game, after Myerson (1991)")
g.append_move(g.root, g.players.chance, ["King", "Queen"])
for node in g.root.children:
    g.append_move(node, "Alice", ["Raise", "Fold"])
g.append_move(g.root.children[0].children[0], "Bob", ["Meet", "Pass"])
g.append_move(g.root.children[1].children[0], "Bob", ["Meet", "Pass"])
# Set inforset
g.set_infoset(g.root.children[0].children[0], g.root.children[1].children[0].infoset)

alice_winsbig = g.add_outcome([2, -2], label="Alice wins big")
alice_wins = g.add_outcome([1, -1], label="Alice wins")
bob_winsbig = g.add_outcome([-2, 2], label="Bob wins big")
bob_wins = g.add_outcome([-1, 1], label="Bob wins")
g.set_outcome(g.root.children[0].children[0].children[0], alice_winsbig)
g.set_outcome(g.root.children[0].children[0].children[1], alice_wins)
g.set_outcome(g.root.children[0].children[1], bob_wins)
g.set_outcome(g.root.children[1].children[0].children[0], bob_winsbig)
g.set_outcome(g.root.children[1].children[0].children[1], alice_wins)
g.set_outcome(g.root.children[1].children[1], bob_wins)
# Save the EFG
g.to_efg('games/one-shot-trust-game-after-kreps-1990.efg')
```

Below is the documentation for several relevant functions in the pygambit library:
1. pygambit.gambit.Game.append_move
Game.append_move(nodes: Node | str | Iterable[Node | str], player: Player | str, actions: List[str]) → None
Add a move for player at terminal nodes. All elements of nodes become part of a new information set, with actions labeled according to actions.

2. pygambit.gambit.Game.add_outcome
Game.add_outcome(payoffs: List | None = None, label: str = '') → Outcome
Add a new outcome to the game.

3. pygambit.gambit.Game.set_outcome
Game.set_outcome(node: Node | str, outcome: Outcome | str | None) → None
Set outcome to be the outcome at node. If outcome is None, the outcome at node is unset.

4. pygambit.gambit.Game.set_infoset
Game.set_infoset(node: Node | str, infoset: Infoset | str) → None
Place node in the information set infoset. node must have the same number of descendants as infoset has actions.
To set the infoset for a node, you can use the following code:
g.set_infoset(node1, node2.infoset)
PLEASE BE AWARE, when more than two nodes need to be grouped in the same information set, you should assign all nodes to the infoset of a single node.
For instance, if you have three nodes, node1, node2, and node3, in the same information set, you can configure the information set as follows:
g.set_infoset(node1, node2.infoset)
g.set_infoset(node3, node2.infoset)

5.pygambit.gambit.Game.set_chance_probs
Game.set_chance_probs(infoset: Infoset | str, probs: Sequence)
Set the action probabilities at chance information set infoset.
For example: 
g.append_move(g.root, g.players.chance, [f"Chance {i}" for i in range(num_choices)])
probabilities = np.random.randint(1,20,n)
total = sum(probabilities)
# Set all the probabilities for the chance node at the same time.
g.set_chance_probs(g.root.infoset, [gbt.Rational(p,total) for p in probabilities])
"""

## Perfect information games with code generation initialization prompt

Before attempting to replicate the full GameInterpreter framework, let's start with perfect information games using only a single code generation initialization prompt.

Get the descriptions of perfect information games from the GitHub repository:

In [7]:
perfect_information_games = get_descriptions("Dataset/Perfect%20Information%20Games")

Found 6 text files in Dataset/Perfect%20Information%20Games:
['Centipede.txt', 'Colonial_Control.txt', 'Market_Entry_Model.txt', 'Nim_(with_five_in_one_pile).txt', 'Simple_Bargaining_Game.txt', 'Tic-Tac-Toe.txt']


In [8]:
display(Markdown(perfect_information_games["Centipede.txt"]))

Consider a game with two players, Alice and Bob, where Alice makes the first move. At the start, Alice has two piles of coins in front of her: one pile with 4 coins and another with 1 coin. Each player has two options on their turn: they can either take the larger pile, giving the smaller pile to the other player, or they can push both piles to the other player. Whenever the piles are pushed across the table, the number of coins in each pile doubles. For instance, if Alice chooses to push on her first turn, the piles of 1 and 4 coins are handed over to Bob, increasing to 2 and 8 coins. Bob can then decide either to take the pile of 8 coins and leave 2 for Alice or to push the piles back to Alice, further doubling them to 4 and 16 coins (The last push will not double the coin). If neither player takes the coins by the end of the game, Alice will receive the pile with the higher value, and Bob will get the one with the lower value. 
The game has four moves in total: Alice moves (take or push), Bob moves (take or push, where the final push also doubles the piles before the game ends), Alice moves again, and finally, Bob moves. All actions are visible to both players.

In [9]:
response = call_copilot("gpt-5",
             prompts["Code_Generation_Initialization.txt"] + "\n\n" +
             perfect_information_games["Centipede.txt"]
             )
response

'Here’s pygambit code implementing the 4-move game; it assumes the final push doubles before end-of-game assignment.\n```python\nimport pygambit as gbt\n\ng = gbt.Game.new_tree(players=["Alice", "Bob"], title="Doubling piles take-or-push (4 moves, perfect information)")\n\n# Move 1: Alice\ng.append_move(g.root, "Alice", ["Take", "Push"])\n# If Alice takes immediately: (4,1)\ng.set_outcome(g.root.children[0], g.add_outcome([4, 1], label="Alice takes at start"))\n\n# Move 2: Bob (after Alice pushes -> piles double to 2 and 8)\ng.append_move(g.root.children[1], "Bob", ["Take", "Push"])\n# Bob takes: (2,8)\ng.set_outcome(g.root.children[1].children[0], g.add_outcome([2, 8], label="Bob takes after A push"))\n\n# Move 3: Alice (after Bob pushes -> piles double to 4 and 16)\ng.append_move(g.root.children[1].children[1], "Alice", ["Take", "Push"])\n# Alice takes: (16,4)\ng.set_outcome(g.root.children[1].children[1].children[0], g.add_outcome([16, 4], label="Alice takes after A push, B push"))\

In [10]:
display(Markdown(response))

Here’s pygambit code implementing the 4-move game; it assumes the final push doubles before end-of-game assignment.
```python
import pygambit as gbt

g = gbt.Game.new_tree(players=["Alice", "Bob"], title="Doubling piles take-or-push (4 moves, perfect information)")

# Move 1: Alice
g.append_move(g.root, "Alice", ["Take", "Push"])
# If Alice takes immediately: (4,1)
g.set_outcome(g.root.children[0], g.add_outcome([4, 1], label="Alice takes at start"))

# Move 2: Bob (after Alice pushes -> piles double to 2 and 8)
g.append_move(g.root.children[1], "Bob", ["Take", "Push"])
# Bob takes: (2,8)
g.set_outcome(g.root.children[1].children[0], g.add_outcome([2, 8], label="Bob takes after A push"))

# Move 3: Alice (after Bob pushes -> piles double to 4 and 16)
g.append_move(g.root.children[1].children[1], "Alice", ["Take", "Push"])
# Alice takes: (16,4)
g.set_outcome(g.root.children[1].children[1].children[0], g.add_outcome([16, 4], label="Alice takes after A push, B push"))

# Move 4: Bob (after Alice pushes -> piles double to 8 and 32)
g.append_move(g.root.children[1].children[1].children[1], "Bob", ["Take", "Push"])
# Bob takes: (8,32)
g.set_outcome(g.root.children[1].children[1].children[1].children[0], g.add_outcome([8, 32], label="Bob takes after two pushes"))
# Final push: piles double to 16 and 64, end-of-game assignment to Alice high, Bob low => (64,16)
g.set_outcome(g.root.children[1].children[1].children[1].children[1], g.add_outcome([64, 16], label="Final push then assign high to Alice"))

# Save the EFG
g.to_efg("games/doubling-piles-take-or-push-4-moves.efg")
```



In [ ]:
# Extract python code block from the response
import re
code_blocks = re.findall(r"```python(.*?)```", response, re.DOTALL)
if code_blocks:
    code = code_blocks[0].strip()
else:
    code = response

In [12]:
exec(code)

In [15]:
generate_png("games/doubling-piles-take-or-push-4-moves.efg")
generate_png("games/centipede_correct.efg")
display(HTML('''
<div style="display: flex; gap: 20px; align-items: center;">
  <img src="doubling-piles-take-or-push-4-moves.png" width="500">
  <img src="centipede_correct.png" width="500">
</div>
'''))